# <center>TensorRT API 基本用法</center>

在上一节课中，我们学习了如何使用 `trtexec` 工具将 ONNX 模型转换为 TensorRT 引擎。`trtexec` 不仅仅是一个方便的命令行工具，它实际上是基于强大的TensorRT API 构建的，这为我们提供了一个直观的例子，展示了API的强大功能和灵活性。

本节课将学习 TensorRT API 的基本用法。通过直接与 API 交互，能够更细致地控制模型的优化和部署过程，从而为各种应用场景定制高性能的推理引擎。更多详细信息和API文档，可以参考 [TensorRT C++ API 文档](https://docs.nvidia.com/deeplearning/tensorrt/api/c_api/index.html) 和 [TensorRT Python API 文档](https://docs.nvidia.com/deeplearning/tensorrt/api/python_api/index.html)。

## 使用 TensorRT API 构建

<center>
    <img src="./course_data/images/builder.png" width=30%>
    <div style="font-size: smaller; color: gray;">
        TensorRT API 构建流程
    </div>
</center>

In [ ]:
import tensorrt as trt

def build_engine(onnx_file, engine_file):
    # 创建 Logger
    logger = trt.Logger(trt.Logger.WARNING)
    # 创建 Builder
    builder = trt.Builder(logger)
    # 创建 Builder Config
    config = builder.create_builder_config()
    # 设置最大工作区大小 为 1 MiB
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 20) # 1 MiB
    # 创建网络定义 要使用 ONNX 解析器导入模型，必须使用 `EXPLICIT_BATCH` 标志。
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    # 创建 ONNX 解析器
    parser = trt.OnnxParser(network, logger)
    # 解析 ONNX 文件
    success = parser.parse_from_file(onnx_file)
    if not success:
        # 打印错误信息并退出
        for idx in range(parser.num_errors):
            print(parser.get_error(idx))
        raise ValueError("Failed to parse ONNX file")
    # 构建并序列化引擎
    serialized_engine = builder.build_serialized_network(network, config)
    # 将序列化后的引擎保存
    with open(engine_file, "wb") as f:
        f.write(serialized_engine)

build_engine("./course_data/models/MNIST/mnist.onnx", "./course_data/models/MNIST/mnist.engine")

<b><font color="red">作业：</font></b>在学会了如何使用 Python API 构建引擎后，请尝试使用 C++ API 实现引擎构建。

## 使用  TensorRT API 部署

<center>
    <img src="./course_data/images/execute.png" width=16%>
    <div style="font-size: smaller; color: gray;">
        TensorRT API 部署流程
    </div>
</center>

In [ ]:
import numpy as np
from course_functions.inference import BaseModel

class MNISTModel(BaseModel):
    def __init__(self, engine_file: str) -> None:
        # 初始化基类，加载TensorRT引擎
        super().__init__(engine_file)
        # 确保有2个tensor（1个输入，1个输出）
        assert len(self.tensors) == 2, "MatrixAddModel expects the engine to have exactly 2 tensors (1 inputs and 1 output)."

    def preprocess(self, data: np.ndarray) -> None:
        """前处理：将输入数据复制到模型的输入张量"""
        # 归一化像素值到 0-1 范围 并 取反
        data = (1 - data / 255).astype(np.float32)
        # 将数据复制到对应的输入张量
        self.tensors[0].memory.host = data

    def postprocess(self) -> int:
        """后处理：从模型的输出张量获取结果"""
        result = self.tensors[1].memory.host[: np.prod(self.tensors[1].shape)].reshape(self.tensors[1].shape)
        return np.argmax(result)

    def predict(self, data: np.ndarray) -> int:
        """推理预测：执行模型的前处理、推理和后处理"""
        self.preprocess(data)
        self.inference()
        return self.postprocess()

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import interact

def mnist_inference(engine_file: str, data_dir: str):
    # 实例化MNIST模型
    model = MNISTModel(engine_file)
    model.warmup()

    def view_infer_result(idx):
        """显示图像及其预测结果"""
        # 加载图像
        image_array = np.array(Image.open(os.path.join(data_dir, f"{idx}.pgm")))
        # 使用模型进行预测
        predicted_label = model.predict(image_array)
        # 显示图像
        plt.imshow(image_array, cmap=plt.cm.gray_r, interpolation='nearest')
        plt.title(f"Actual: {idx} , Predicted: {predicted_label}")
        plt.axis('off')
        plt.show()

    # 创建交互式界面，允许用户选择图像索引
    interact(view_infer_result, idx=(0, 9))

In [ ]:
mnist_inference("./course_data/models/MNIST/mnist.engine", "./course_data/models/MNIST")

<b><font color="red">作业：</font></b>在学会了如何使用 Python API 部署后，请尝试使用 C++ API 实现部署。

## <center>总结</center>

通过本节课的学习，我们掌握了 TensorRT API 的基本操作，并能够使用 Python API 构建和部署高性能的推理引擎。同时，我们也布置了作业，希望您能够尝试使用 C++ API 实现引擎构建和部署，以加深对 TensorRT API 的理解和掌握。在后续课程中，我们将继续深入学习 TensorRT 的高级功能和优化技巧，帮助您进一步提升模型的性能和效率。